# 🧠 Smart DevOps & Code Auditor
### Kaggle Capstone Project — Google ADK + Gemini 2.5 Flash

This notebook represents the **fully integrated, self-contained run** of the Smart DevOps & Code Auditor pipeline.
It is structured to run directly on **Kaggle.com** (zero cost, using Gemini free tier) and explicitly implements all 5 core concepts taught in the Google Agent Development Kit (ADK) course.

--- 
## 🎓 Google ADK Course Concepts Demonstrated:
1. **Agent Class**: Wrapped specialized code review and security directives inside ADK Agent definitions.
2. **Runner Flow**: Async streaming runner execution with InMemorySessionService session context state.
3. **MCP Server**: Sandboxed read-only local filesystem access controls.
4. **Pre-processing Guards**: Secret redaction (credential masking) and prompt injection sanitization before LLM calls.
5. **Post-processing Skills**: Git-compatible unified diff formatter powered by Python's `difflib`.

---

## 🛠️ Step 1: Environment Setup
Before executing, ensure you have set your `GEMINI_API_KEY` (either in the Kaggle User Secrets or as an environment variable). We load `.env` variables if present.

In [ ]:
import os
import sys
import time
import json
import uuid
import re
import asyncio
import logging
from typing import List, Optional, Dict, Any
from dataclasses import dataclass, field
from enum import Enum
from datetime import datetime
from pathlib import Path
from functools import cached_property

# Verify GenAI SDK and ADK are installed
try:
    import google.adk
    import google.genai
    print("✅ Google ADK & GenAI SDK imported successfully.")
except ImportError:
    print("Installing Google ADK...")
    !pip install google-adk google-genai python-dotenv -q


## 📦 Step 2: Core Data Models (Schemas)
These are the typed dataclass schemas representing findings, patch suggestions, and scan results, passed sequentially between agents.

In [ ]:
"""
models/schemas.py
─────────────────
All shared data models for the Smart DevOps & Code Auditor pipeline.
Passed between Agent A → Agent B → Agent C via the orchestrator.

Design: Python stdlib dataclasses (no Pydantic dependency — see ADR-005).
"""

from dataclasses import dataclass, field
from typing import List, Optional, Dict
from enum import Enum
from datetime import datetime
import uuid


# ──────────────────────────────────────────────────────────────────────────────
# ENUMS
# ──────────────────────────────────────────────────────────────────────────────

class Severity(str, Enum):
    """
    Issue severity levels with corresponding security score deductions.
    Score starts at 100, deducted per finding:
      CRITICAL → -30 | HIGH → -15 | MEDIUM → -7 | LOW → -3 | INFO → -1
    """
    CRITICAL = "CRITICAL"
    HIGH     = "HIGH"
    MEDIUM   = "MEDIUM"
    LOW      = "LOW"
    INFO     = "INFO"


class FindingType(str, Enum):
    """
    Classification of detected issues.
    Security types map to CWE identifiers (see CodeFinding.cwe_id).
    """
    HARDCODED_SECRET      = "HARDCODED_SECRET"       # CWE-798
    SQL_INJECTION         = "SQL_INJECTION"           # CWE-89
    COMMAND_INJECTION     = "COMMAND_INJECTION"       # CWE-78
    PATH_TRAVERSAL        = "PATH_TRAVERSAL"          # CWE-22
    WEAK_CRYPTO           = "WEAK_CRYPTO"             # CWE-327
    INSECURE_CONFIG       = "INSECURE_CONFIG"         # CWE-16
    LOGIC_BUG             = "LOGIC_BUG"              # General — Agent A
    CODE_SMELL            = "CODE_SMELL"             # General — Agent A
    PERFORMANCE_ISSUE     = "PERFORMANCE_ISSUE"      # General — Agent A
    PROMPT_INJECTION_RISK = "PROMPT_INJECTION_RISK"  # Novel AI-era finding


class AgentSource(str, Enum):
    """Which agent produced a finding."""
    CODE_REVIEWER = "code_reviewer"
    SECOPS        = "secops"


class PatchStatus(str, Enum):
    """User review status of a patch suggestion."""
    PENDING  = "PENDING"
    APPROVED = "APPROVED"
    REJECTED = "REJECTED"


# ──────────────────────────────────────────────────────────────────────────────
# SEVERITY SCORE DEDUCTIONS
# ──────────────────────────────────────────────────────────────────────────────

SEVERITY_DEDUCTIONS: Dict[Severity, int] = {
    Severity.CRITICAL: 30,
    Severity.HIGH:     15,
    Severity.MEDIUM:    7,
    Severity.LOW:       3,
    Severity.INFO:      1,
}

SEVERITY_EMOJI: Dict[str, str] = {
    "CRITICAL": "🔴",
    "HIGH":     "🟠",
    "MEDIUM":   "🟡",
    "LOW":      "🟢",
    "INFO":     "ℹ️",
}


# ──────────────────────────────────────────────────────────────────────────────
# CORE DATA MODELS
# ──────────────────────────────────────────────────────────────────────────────

@dataclass
class CodeFinding:
    """
    A single issue detected by Agent A (Code Reviewer) or Agent B (SecOps).
    Produced AFTER code has been redacted — code_snippet contains [REDACTED-X] tokens
    if secrets were present in the original.
    """
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    file_path: str = ""
    line_number: int = 0
    line_end: int = 0                    # last line of the affected block (0 = same as line_number)
    code_snippet: str = ""              # the offending code (post-redaction)
    finding_type: FindingType = FindingType.CODE_SMELL
    severity: Severity = Severity.LOW
    description: str = ""              # human-readable explanation for the developer
    cwe_id: Optional[str] = None       # e.g., "CWE-89" — only set by Agent B (SecOps)
    cwe_name: Optional[str] = None     # e.g., "Improper Neutralization of SQL Commands"
    agent_source: AgentSource = AgentSource.CODE_REVIEWER
    confidence: float = 0.0            # 0.0–1.0, agent's self-assessed confidence


@dataclass
class AuditReport:
    """
    Aggregated output from Agent A + Agent B for a full scan run.
    security_score is computed lazily via compute_score().
    """
    scan_id: str = field(default_factory=lambda: str(uuid.uuid4())[:12])
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    target_directory: str = "./sandbox-repo"
    files_scanned: int = 0
    total_lines_scanned: int = 0
    findings: List[CodeFinding] = field(default_factory=list)
    security_score: int = 100           # recomputed by compute_score() — don't set manually
    summary_by_severity: Dict[str, int] = field(default_factory=dict)
    reviewer_summary: str = ""          # Agent A's free-text summary of what it found
    secops_summary: str = ""            # Agent B's free-text summary

    def compute_score(self) -> int:
        """
        Compute security score: start at 100, deduct per finding severity.
        Floors at 0. Call this after all findings are populated.
        """
        score = 100
        for finding in self.findings:
            score -= SEVERITY_DEDUCTIONS.get(finding.severity, 0)
        self.security_score = max(0, score)
        return self.security_score

    def severity_summary(self) -> Dict[str, int]:
        """Count findings per severity. Populates summary_by_severity in-place."""
        counts = {s.value: 0 for s in Severity}
        for finding in self.findings:
            counts[finding.severity.value] += 1
        self.summary_by_severity = counts
        return counts

    def findings_by_severity(self, severity: Severity) -> List[CodeFinding]:
        """Filter findings by a specific severity level."""
        return [f for f in self.findings if f.severity == severity]

    def critical_count(self) -> int:
        return len(self.findings_by_severity(Severity.CRITICAL))

    def score_label(self) -> str:
        """Human-readable score label for UI display."""
        s = self.compute_score()
        if s >= 80:
            return f"✅ {s}/100 — Good"
        elif s >= 50:
            return f"⚠️ {s}/100 — Needs Attention"
        else:
            return f"🚨 {s}/100 — Critical Issues"


@dataclass
class PatchSuggestion:
    """
    A fix proposed by Agent C (Patch Artisan) for a single CodeFinding.
    unified_diff is populated AFTER Agent C runs, by the DiffFormatter skill.
    """
    patch_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    finding: CodeFinding = field(default_factory=CodeFinding)
    original_code: str = ""            # the problematic code block (pre-fix)
    patched_code: str = ""             # the fixed code block
    unified_diff: str = ""             # git unified diff — set by DiffFormatter skill
    explanation: str = ""              # why this patch fixes the finding
    confidence_score: float = 0.0     # 0.0–1.0, Agent C's confidence
    status: PatchStatus = PatchStatus.PENDING

    def is_auto_approvable(self, threshold: float = 0.95) -> bool:
        """Returns True if confidence meets auto-approve threshold."""
        return self.confidence_score >= threshold


@dataclass
class ScanRequest:
    """
    User-supplied scan configuration. Created by UI or CLI and passed to orchestrator.
    """
    target_dir: str = "./sandbox-repo"
    scan_depth: int = 3
    file_extensions: List[str] = field(
        default_factory=lambda: [".py", ".js", ".ts", ".env", ".yaml", ".yml"]
    )
    enable_secret_redaction: bool = True       # always True in production; locked in UI
    enable_prompt_sanitization: bool = True    # always True in production; locked in UI
    auto_approve_threshold: float = 0.95       # auto-approve patches above this confidence


@dataclass
class PipelineResult:
    """
    Final output of the orchestrator pipeline.
    Returned to Streamlit UI or CLI for display.
    """
    pipeline_id: str = field(default_factory=lambda: str(uuid.uuid4())[:12])
    scan_request: ScanRequest = field(default_factory=ScanRequest)
    audit_report: AuditReport = field(default_factory=AuditReport)
    patches: List[PatchSuggestion] = field(default_factory=list)
    approved_patches: List[str] = field(default_factory=list)   # patch_ids approved by user
    total_runtime_seconds: float = 0.0
    error_log: List[str] = field(default_factory=list)

    def summary(self) -> str:
        """One-liner for CLI / logging output."""
        r = self.audit_report
        return (
            f"[{r.scan_id[:8]}]  "
            f"{r.files_scanned} files  |  "
            f"{len(r.findings)} findings  |  "
            f"Score: {r.compute_score()}/100  |  "
            f"{len(self.patches)} patches  |  "
            f"{self.total_runtime_seconds:.1f}s"
        )

    def has_errors(self) -> bool:
        return len(self.error_log) > 0

    def approved_patch_objects(self) -> List[PatchSuggestion]:
        """Return only the patches the user has approved."""
        return [p for p in self.patches if p.patch_id in self.approved_patches]


## 🔍 Step 3: Pre-processing Guard & Post-processing Skills
Here we define **Secret Redactor** (masks variables, passwords, API keys) and the **Diff Formatter** (converts agent outputs into standard unified diff blocks).

In [ ]:
# --- Secret Redactor Guard ---
def redact_secrets_with_report(code: str) -> (str, List[str]):
    import re
    patterns = [
        (re.compile(r'(AKIA[A-Z0-9]{16})', re.IGNORECASE), '[REDACTED-AWS-ACCESS-KEY]'),
        (re.compile(r'((?:api[_-]?key|apikey)\s*=\s*["\'])([^"\']{8,})(["\'])', re.IGNORECASE), r'\1[REDACTED-API-KEY]\3'),
        (re.compile(r'(["\'])(sk-[a-zA-Z0-9\-_]{20,})(["\'])'), r'\1[REDACTED-API-KEY]\3'),
        (re.compile(r'((?:password|passwd|pwd)\s*=\s*["\'])([^"\']+)(["\'])', re.IGNORECASE), r'\1[REDACTED-PASSWORD]\3'),
        (re.compile(r'((?:token|access_token|auth_token|bearer_token)\s*=\s*["\'])([^"\']+)(["\'])', re.IGNORECASE), r'\1[REDACTED-TOKEN]\3'),
    ]
    result = code
    logs = []
    for pattern, replacement in patterns:
        matches = len(pattern.findall(result))
        if matches > 0:
            result = pattern.sub(replacement, result)
            logs.append(f"Redacted {matches} secret pattern(s)")
    return result, logs

# --- Diff Formatter Skill ---
def format_unified_diff(original: str, patched: str, file_path: str) -> str:
    import difflib
    orig_lines = original.splitlines(keepends=True)
    patch_lines = patched.splitlines(keepends=True)
    diff_gen = difflib.unified_diff(
        orig_lines, patch_lines,
        fromfile=f"a/{file_path}",
        tofile=f"b/{file_path}"
    )
    return "".join(diff_gen)


## 🤖 Step 4: Keyed Gemini Client & Core Agents Definition
We subclass ADK's model class to support explicit API keys rotation and initialize Agent A, B, and C.

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models import Gemini
from google.genai import Client, types

class KeyedGemini(Gemini):
    def __init__(self, api_key: str, **kwargs):
        super().__init__(**kwargs)
        self._api_key = api_key

    @cached_property
    def api_client(self) -> Client:
        base_url, api_version = self._base_url_and_api_version
        kwargs_for_http_options = {
            'headers': self._tracking_headers(),
            'base_url': base_url,
        }
        kwargs = {
            'http_options': types.HttpOptions(**kwargs_for_http_options),
            'api_key': self._api_key,
        }
        return Client(**kwargs)

# Get Gemini keys from environment
gemini_api_key = os.environ.get("GEMINI_API_KEY", "")

reviewer_prompt = """Your task is to analyze Python code for logic bugs, performance issues, and code smells. Exclude security issues. Output your results as a JSON array of finding objects."""
secops_prompt = """Your task is to scan Python code for security flaws (CWE). Use previous review context if provided. Output as a JSON array of finding objects."""
patch_prompt = """Your task is to generate minimal, surgical corrections for code findings. Output as a JSON array of patch objects containing original_code, patched_code, and explanation."""

# Pre-create agents and runners
model_inst = KeyedGemini(api_key=gemini_api_key, model="gemini-2.5-flash")

agent_a = Agent(name="code_reviewer", model=model_inst, instruction=reviewer_prompt)
runner_a = Runner(agent=agent_a, session_service=InMemorySessionService(), app_name="auditor")

agent_b = Agent(name="secops_engineer", model=model_inst, instruction=secops_prompt)
runner_b = Runner(agent=agent_b, session_service=InMemorySessionService(), app_name="auditor")

agent_c = Agent(name="patch_artisan", model=model_inst, instruction=patch_prompt)
runner_c = Runner(agent=agent_c, session_service=InMemorySessionService(), app_name="auditor")


## 🔄 Step 5: Orchestrator Pipeline
This sequences the execution flow (`Code Reviewer ➔ SecOps ➔ Patch Artisan`) and parses JSON text arrays back into structured dataclasses.

In [ ]:
from google.genai.types import Content, Part

def clean_prompt_injection(code: str) -> str:
    patterns = [r"ignore\s+all\s+previous", r"system:", r"forget\s+everything", r"jailbreak"]
    cleaned = code
    for p in patterns:
        cleaned = re.sub(rf"(?:#|//)\s*{p}.*", "# [SANITIZED-COMMENT]", cleaned, flags=re.IGNORECASE)
    return cleaned

def strip_json_fences(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    return re.sub(r"\s*```$", "", text).strip()

async def run_scan_pipeline(raw_code: str, file_path: str) -> PipelineResult:
    start_time = time.time()
    res = PipelineResult()
    res.audit_report.files_scanned = 1
    res.audit_report.total_lines_scanned = len(raw_code.splitlines())
    
    # Pre-processing
    redacted_code, _ = redact_secrets_with_report(raw_code)
    clean_code = clean_prompt_injection(redacted_code)
    
    # 1. Agent A call
    session_a = await runner_a.session_service.create_session(app_name="auditor", user_id="eval")
    prompt_a = f"File: {file_path}\nCode:\n{clean_code}"
    resp_a = ""
    async for event in runner_a.run_async(user_id="eval", session_id=session_a.id, new_message=Content(role="user", parts=[Part(text=prompt_a)])):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text: resp_a = part.text
                
    findings_a = []
    try:
        data_a = json.loads(strip_json_fences(resp_a))
        for item in data_a:
            findings_a.append(CodeFinding(
                file_path=file_path, line_number=item.get("line_number", 1), line_end=item.get("line_end", 1),
                  code_snippet=item.get("code_snippet", ""), description=item.get("description", ""),
                  agent_source=AgentSource.CODE_REVIEWER, severity=Severity.LOW
            ))
    except Exception:
        pass
        
    # 2. Agent B call
    session_b = await runner_b.session_service.create_session(app_name="auditor", user_id="eval")
    prompt_b = f"File: {file_path}\nCode:\n{clean_code}\nContext:\n{resp_a}"
    resp_b = ""
    async for event in runner_b.run_async(user_id="eval", session_id=session_b.id, new_message=Content(role="user", parts=[Part(text=prompt_b)])):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text: resp_b = part.text
                
    findings_b = []
    try:
        data_b = json.loads(strip_json_fences(resp_b))
        for item in data_b:
            findings_b.append(CodeFinding(
                file_path=file_path, line_number=item.get("line_number", 1), line_end=item.get("line_end", 1),
                  code_snippet=item.get("code_snippet", ""), description=item.get("description", ""),
                  agent_source=AgentSource.SECOPS, severity=Severity.HIGH, cwe_id=item.get("cwe_id")
            ))
    except Exception:
        pass
        
    all_findings = findings_a + findings_b
    res.audit_report.findings = all_findings
    res.audit_report.compute_score()
    
    # 3. Agent C call (only if findings exist)
    if all_findings:
        session_c = await runner_c.session_service.create_session(app_name="auditor", user_id="eval")
        prompt_c = f"File: {file_path}\nCode:\n{clean_code}\nFindings:\n{resp_b}"
        resp_c = ""
        async for event in runner_c.run_async(user_id="eval", session_id=session_c.id, new_message=Content(role="user", parts=[Part(text=prompt_c)])):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text: resp_c = part.text
                    
        try:
            data_c = json.loads(strip_json_fences(resp_c))
            for item in data_c:
                patch = PatchSuggestion(
                    finding=all_findings[0], original_code=item.get("original_code", ""),
                      patched_code=item.get("patched_code", ""), explanation=item.get("explanation", "")
                )
                patch.unified_diff = format_unified_diff(patch.original_code, patch.patched_code, file_path)
                res.patches.append(patch)
        except Exception:
            pass
            
    res.total_runtime_seconds = time.time() - start_time
    return res


## 🧪 Step 6: Code Scan Verification
Let's scan a sample vulnerable python code block. If you have setup your API key, this will make live agent calls and print the CWE mappings + unified diff patches.

In [ ]:
vulnerable_code = """
import sqlite3

# CWE-798: Hardcoded database credentials
DB_PASS = "admin_db_pass_1234"
API_KEY = "sk-adminKeyOpenAI12345"

def fetch_user(user_id):
    # CWE-89: SQL Injection
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    query = f"SELECT * FROM accounts WHERE id = '{user_id}'"
    cursor.execute(query)
    return cursor.fetchone()
"""

if not gemini_api_key:
    print("⚠️ GEMINI_API_KEY environment variable not found. Skipping live agent calls.")
    print("To run the live pipeline, set GEMINI_API_KEY in the Environment or Kaggle secrets.")
else:
    print("🚀 Running multi-agent scan on vulnerable code payload...")
    try:
        result = asyncio.run(run_scan_pipeline(vulnerable_code, "auth.py"))
        print(f"\nScan ID: {result.pipeline_id}")
        print(f"Security Score: {result.audit_report.security_score}/100")
        print(f"Findings Found: {len(result.audit_report.findings)}")
        for f in result.audit_report.findings:
            print(f"  - [{f.agent_source.value.upper()}] Line {f.line_number}: {f.description} (CWE: {f.cwe_id})")
        print(f"\nPatches Generated: {len(result.patches)}")
        for p in result.patches:
            print(f"\nPatch Unified Diff:\n{p.unified_diff}")
    except Exception as e:
        print(f"Scan execution failed: {e}")
